
# Sampling desde el resultado de limpieza (`dataspark.cleansing` → `dataspark.sampling`)

Este cuaderno continúa el flujo: **cargar datos → limpiar → muestrear**.

Objetivo:
1. Retomar un DataFrame ya limpiado con `DataCleaner`.
2. Probar todas las funciones de `Sampler`.
3. Explicar claramente los parámetros principales de cada método.



## Dataset (ligero, desde Internet)

Se usa Titanic desde URL pública:
- `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv`

Si el entorno no permite internet, se usa fallback local (`sklearn.load_iris`) para que el cuaderno siga siendo ejecutable.


In [ ]:

import pandas as pd
import numpy as np

from dataspark.cleansing import DataCleaner
from dataspark.sampling import Sampler

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


In [ ]:

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

try:
    raw_df = pd.read_csv(url)
    source_name = f"Titanic (URL): {url}"
except Exception as exc:
    from sklearn.datasets import load_iris
    iris = load_iris(as_frame=True)
    raw_df = iris.frame
    source_name = "Fallback local: sklearn.datasets.load_iris()"
    print("No se pudo descargar la URL por restricciones de red:", exc)

print("Fuente:", source_name)
print("Shape original:", raw_df.shape)
raw_df.head()


## 1) Preparar datos con ruido y aplicar `DataCleaner`


In [ ]:

df = raw_df.copy()

# Para demostrar limpieza de nombres, espacios y nulos
rename_map = {
    "survived": " Survived ",
    "fare": "Fare($)",
    "embarked": "Embarked Port"
}
df = df.rename(columns=rename_map)

if "class" in df.columns:
    df["class"] = df["class"].astype(str).map(lambda x: f"  {x}  ")

rng = np.random.default_rng(123)
idx = rng.choice(len(df), size=min(30, len(df)), replace=False)
if "Fare($)" in df.columns:
    df.loc[idx[:10], "Fare($)"] = np.nan
if "age" in df.columns:
    df.loc[idx[10:20], "age"] = np.nan
if "Embarked Port" in df.columns:
    df.loc[idx[20:30], "Embarked Port"] = np.nan

print("Nulos antes de limpiar:")
print(df.isna().sum().sort_values(ascending=False).head(10))


In [ ]:

cleaner = DataCleaner(missing_strategy="median", standardize_columns=True, strip_whitespace=True)
df_clean = cleaner.fit_transform(df)

print("Shape limpio:", df_clean.shape)
print("Nulos después de limpiar:", df_clean.isna().sum().sum())
df_clean.head()



## 2) `Sampler.__init__(random_state)`

**Parámetro clave**
- `random_state`: semilla para reproducibilidad. Si repites el cuaderno con la misma semilla, obtendrás la misma selección (salvo cambios en datos de entrada).


In [ ]:

sampler = Sampler(random_state=42)
sampler



## 3) `stratified_sample(df, stratum_col, n=None, frac=None)`

**Parámetros**
- `stratum_col`: columna de estrato (ej. clase, categoría, segmento).
- `n`: tamaño total objetivo (asignación proporcional por estrato).
- `frac`: fracción por estrato (alternativa a `n`).

> Debes pasar `n` o `frac`.


In [ ]:

# Elegimos una columna categórica disponible
candidate_strata = [c for c in ["class", "sex", "embarked_port", "species"] if c in df_clean.columns]
stratum_col = candidate_strata[0]
print("Estrato usado:", stratum_col)

strat_n = sampler.stratified_sample(df_clean, stratum_col=stratum_col, n=min(120, len(df_clean)))
strat_frac = sampler.stratified_sample(df_clean, stratum_col=stratum_col, frac=0.2)

print("strat_n shape:", strat_n.shape)
print("strat_frac shape:", strat_frac.shape)
strat_n[stratum_col].value_counts(normalize=True).head()



## 4) `cluster_sample(df, cluster_col, n_clusters)`

**Parámetros**
- `cluster_col`: etiqueta de conglomerado.
- `n_clusters`: número de conglomerados a seleccionar.

Devuelve todas las filas de los conglomerados elegidos.


In [ ]:

cluster_col = stratum_col
clustered = sampler.cluster_sample(df_clean, cluster_col=cluster_col, n_clusters=2)

print("Cluster sample shape:", clustered.shape)
print("Clusters seleccionados:", clustered[cluster_col].unique())



## 5) `systematic_sample(df, k=None, n=None)`

**Parámetros**
- `k`: salto fijo (cada k-ésima fila).
- `n`: tamaño deseado; el método calcula `k` automáticamente.

Usa un inicio aleatorio interno para evitar sesgo por posición inicial.


In [ ]:

sys_k = sampler.systematic_sample(df_clean, k=5)
sys_n = sampler.systematic_sample(df_clean, n=min(100, len(df_clean)))

print("systematic k=5:", sys_k.shape)
print("systematic n=100 aprox:", sys_n.shape)



## 6) `reservoir_sample(df, n)`

**Parámetro**
- `n`: tamaño del reservorio.

Útil para flujos de datos grandes/continuos donde no quieres cargar todo en memoria para muestrear.


In [ ]:

reservoir = sampler.reservoir_sample(df_clean, n=min(80, len(df_clean)))
print("Reservoir shape:", reservoir.shape)
reservoir.head()



## 7) `bootstrap_sample(df, n_samples=1000, statistic="mean", column=None)`

**Parámetros**
- `n_samples`: número de remuestreos bootstrap.
- `statistic`: estadístico NumPy a usar (`mean`, `median`, `std`, etc.).
- `column`: columna numérica objetivo.

Devuelve media bootstrap del estadístico, desviación estándar e intervalo de confianza percentil.


In [ ]:

numeric_cols = df_clean.select_dtypes(include="number").columns.tolist()
col = numeric_cols[0]

boot = sampler.bootstrap_sample(
    df_clean,
    n_samples=300,
    statistic="mean",
    column=col,
)

boot



## 8) `sample_size_calculator(population_size, confidence_level=0.95, margin_of_error=0.05, proportion=0.5)`

**Parámetros**
- `population_size`: tamaño total de población.
- `confidence_level`: nivel de confianza deseado.
- `margin_of_error`: error máximo tolerado.
- `proportion`: proporción esperada (0.5 es caso conservador).


In [ ]:

sample_size_info = sampler.sample_size_calculator(
    population_size=max(len(df_clean), 1000),
    confidence_level=0.95,
    margin_of_error=0.05,
    proportion=0.5,
)

sample_size_info



## 9) Resumen

En este cuaderno:
- retomamos datos desde la fase de limpieza (`DataCleaner`),
- aplicamos todas las funciones públicas del módulo `sampling`,
- explicamos los parámetros para que puedas adaptar cada método a tus escenarios.
